In [0]:
-- =====================================================================
-- Coffee Shop Sales Analysis
-- Case Study 1 — Databricks SQL
-- =====================================================================
-- Source table: coffee_shop.cshop_db.coffeeshopdataset
-- =====================================================================


-- =====================================================================
-- SECTION 1: DATA QUALITY CHECKS
-- =====================================================================

-- Preview the raw data
SELECT *
FROM coffee_shop.cshop_db.coffeeshopdataset
LIMIT 10;

-- Check all columns are in the correct format
-- unit_price needs to be changed (commas used as decimal separators, e.g. '3,1' -> 3.1)
SELECT CAST(REPLACE(unit_price, ',', '.') AS DOUBLE) AS unit_price
FROM coffee_shop.cshop_db.coffeeshopdataset;

-- Check for duplicate rows
SELECT (*),
       COUNT(*)
FROM coffee_shop.cshop_db.coffeeshopdataset
GROUP BY ALL;

-- Check for NULL values
-- Result: no NULL values found
SELECT *
FROM coffee_shop.cshop_db.coffeeshopdataset
WHERE transaction_id IS NULL
   OR transaction_date IS NULL
   OR transaction_time IS NULL
   OR transaction_qty IS NULL
   OR store_id IS NULL
   OR store_location IS NULL
   OR unit_price IS NULL
   OR product_category IS NULL
   OR product_detail IS NULL;


-- =====================================================================
-- SECTION 2: BUSINESS QUESTIONS
-- (all columns included in the analysis as required)
-- =====================================================================

-- 2.1 Total revenue
SELECT SUM(transaction_qty * CAST(REPLACE(unit_price, ',', '.') AS DECIMAL(10, 2))) AS total_revenue
FROM coffee_shop.cshop_db.coffeeshopdataset;

-- 2.2 Total revenue per product category
SELECT product_category,
       SUM(transaction_qty * CAST(REPLACE(unit_price, ',', '.') AS DECIMAL(10, 2))) AS total_revenue
FROM coffee_shop.cshop_db.coffeeshopdataset
GROUP BY 1;

-- 2.3 Total revenue per product category and product_detail
SELECT product_category,
       product_detail,
       SUM(transaction_qty * CAST(REPLACE(unit_price, ',', '.') AS DECIMAL(10, 2))) AS total_revenue
FROM coffee_shop.cshop_db.coffeeshopdataset
GROUP BY 1, 2;

-- Check the earliest and latest transaction times (supports the time-of-day bucketing below)
SELECT MIN(transaction_time),
       MAX(transaction_time)
FROM coffee_shop.cshop_db.coffeeshopdataset;

-- 2.4 Total revenue per product type
SELECT product_type,
       SUM(transaction_qty * CAST(REPLACE(unit_price, ',', '.') AS DECIMAL(10, 2))) AS total_revenue
FROM coffee_shop.cshop_db.coffeeshopdataset
GROUP BY ALL;

-- 2.5 Total revenue per month name
SELECT MONTHNAME(transaction_date) AS month_name,
       SUM(transaction_qty * CAST(REPLACE(unit_price, ',', '.') AS DECIMAL(10, 2))) AS total_revenue
FROM coffee_shop.cshop_db.coffeeshopdataset
GROUP BY ALL;

-- 2.6 Total revenue based on time of day, by product category and month
-- Time buckets:
--   06:00 - 11:59  -> Morning
--   12:00 - 16:59  -> Afternoon
--   17:00 - 19:59  -> Evening
--   else           -> Night
SELECT product_category,
       MONTHNAME(transaction_date) AS month_name,
       CASE
           WHEN DATE_FORMAT(transaction_time, 'HH:mm:ss') BETWEEN '06:00:00' AND '11:59:59' THEN 'Morning'
           WHEN DATE_FORMAT(transaction_time, 'HH:mm:ss') BETWEEN '12:00:00' AND '16:59:59' THEN 'Afternoon'
           WHEN DATE_FORMAT(transaction_time, 'HH:mm:ss') BETWEEN '17:00:00' AND '19:59:59' THEN 'Evening'
           ELSE 'Night'
       END AS time_bucket,
       SUM(transaction_qty * CAST(REPLACE(unit_price, ',', '.') AS DECIMAL(10, 2))) AS total_revenue
FROM coffee_shop.cshop_db.coffeeshopdataset
GROUP BY ALL;

-- 2.7 Total revenue based on time of day, by product category only
SELECT product_category,
       CASE
           WHEN DATE_FORMAT(transaction_time, 'HH:mm:ss') BETWEEN '06:00:00' AND '11:59:59' THEN 'Morning'
           WHEN DATE_FORMAT(transaction_time, 'HH:mm:ss') BETWEEN '12:00:00' AND '16:59:59' THEN 'Afternoon'
           WHEN DATE_FORMAT(transaction_time, 'HH:mm:ss') BETWEEN '17:00:00' AND '19:59:59' THEN 'Evening'
           ELSE 'Night'
       END AS time_bucket,
       SUM(transaction_qty * CAST(REPLACE(unit_price, ',', '.') AS DECIMAL(10, 2))) AS total_revenue
FROM coffee_shop.cshop_db.coffeeshopdataset
GROUP BY ALL;

-- 2.8 Date extraction: month name, month id, day name, day number
-- month_id assists with sorting months in calendar order
SELECT transaction_date,
       MONTHNAME(transaction_date) AS month_name,
       DATE_FORMAT(transaction_date, 'yyyy-MMM') AS month_id,
       DAYNAME(transaction_date) AS day_name,
       DAYOFWEEK(transaction_date) AS day_number
FROM coffee_shop.cshop_db.coffeeshopdataset;


-- =====================================================================
-- SECTION 3: FINAL COMBINED TABLE
-- Brings all business questions above into a single analysis-ready table
-- =====================================================================

SELECT DATE_FORMAT(transaction_date, 'yyyy-MM-dd') AS transaction_date,
       MONTHNAME(transaction_date) AS month_name,
       DATE_FORMAT(transaction_date, 'yyyy-MMM') AS month_id,
       DAYNAME(transaction_date) AS day_name,
       DAYOFWEEK(transaction_date) AS day_number,
       COUNT(product_id) AS trans_count,
       COUNT(product_id) AS products_sold,
       product_category,
       product_detail,
       product_type,
       store_location,
       CASE
           WHEN DATE_FORMAT(transaction_time, 'HH:mm:ss') BETWEEN '06:00:00' AND '11:59:59' THEN 'Morning'
           WHEN DATE_FORMAT(transaction_time, 'HH:mm:ss') BETWEEN '12:00:00' AND '16:59:59' THEN 'Afternoon'
           WHEN DATE_FORMAT(transaction_time, 'HH:mm:ss') BETWEEN '17:00:00' AND '19:59:59' THEN 'Evening'
           ELSE 'Night'
       END AS time_bucket,
       SUM(transaction_qty * CAST(REPLACE(unit_price, ',', '.') AS DECIMAL(10, 2))) AS total_revenue
FROM coffee_shop.cshop_db.coffeeshopdataset
GROUP BY ALL;